**Work in progress, archived.** Batch version intended to loop GMM clustering over a folder of monthly pickles. It currently fails (see the recorded `IndexError`, and an undefined `land_idx` in the last cell), so it is kept only as a record of the direction taken. The working pipeline is `01_saphir_preprocessing_dcc_gmm.ipynb`.


In [ ]:
import os
import h5py as h5
import numpy as np
import grid_func  # Assuming grid_func.py is available in your program folder
import matplotlib.pyplot as plt
import pickle
from sklearn.mixture import GaussianMixture

# Function to process each pickle file
def process_pickle_file(pickle_file_path):
    # Open the pickle file and read its content
    with open(pickle_file_path, 'rb') as pickle_file:
        data = pickle_file.read()

    # Deserialize the content of the pickle file
    spdct = pickle.loads(data)

    # Compute flag and level using grid_func.dcc_flag()
    flag, level = grid_func.dcc_flag(spdct['bt1sp'], spdct['bt2sp'], spdct['bt3sp'], spdct['bt4sp'], spdct['bt5sp'], spdct['stznsp'])
    spdct['dcc'] = flag
    spdct['level'] = level

    # Filter data where stznsp and bt2sp are not equal to -9999.0
    noval = -9999.0
    idx = np.where((spdct['stznsp'] != noval) * (spdct['bt2sp'] != noval))

    # Create train and test datasets
    train = {}
    train['stznsp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train['bt1sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train['bt2sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train['bt3sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train['bt4sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train['bt5sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train['mask'] = np.ones(np.shape(spdct['stznsp'])) * noval

    train['stznsp'] = spdct['stznsp'][idx]
    train['bt1sp'] = spdct['bt1sp'][idx]
    train['bt2sp'] = spdct['bt2sp'][idx]
    train['bt3sp'] = spdct['bt3sp'][idx]
    train['bt4sp'] = spdct['bt4sp'][idx]
    train['bt5sp'] = spdct['bt5sp'][idx]
    train['mask'] = spdct['mask'][idx]

    test = {}
    test['dcc'] = spdct['dcc'][idx]
    test['level'] = spdct['level'][idx]

    # Prepare features for Gaussian Mixture Model
    features = ['stznsp', 'bt1sp', 'bt2sp', 'bt3sp', 'bt4sp', 'bt5sp', 'mask']
    X_train = np.column_stack([train[feature] for feature in features])

    # Initialize and train Gaussian Mixture Model
    n_components = 2  # Number of components (clusters)
    gmm = GaussianMixture(n_components=n_components, covariance_type='full')
    gmm.fit(X_train)

    # Predict cluster assignments for the training data
    train_predict = gmm.predict(X_train)

    # Prepare predicted data
    predicted = np.ones(np.shape(spdct['stznsp'])) * noval
    flat_idx = [(idx[0][i], idx[1][i]) for i in range(len(idx[0]))]
    for (i, j), label in zip(flat_idx, train_predict):
        predicted[i, j] = label

    # Mask the predicted data to handle -9999 values
    masked_data = np.ma.masked_equal(predicted, noval)

    # Plotting
    plt.imshow(masked_data, cmap='viridis', interpolation='nearest')

    # Create colorbar
    cbar = plt.colorbar()
    cbar.set_ticks([0, 1])  # Set ticks for valid data (0 and 1)
    cbar.set_ticklabels(['0', '1'])  # Set labels for ticks

    # Extract file name from path
    file_name = os.path.basename(pickle_file_path)
    plt.title(f"Cluster Plot - {file_name}")

    # Save plot to a file (adjust the saving path as needed)
    save_path = os.path.join('/content/Plots_Jan2014', f"{os.path.splitext(file_name)[0]}.png")
    plt.savefig(save_path)
    plt.close()  # Close plot to free up memory

# Specify the path to the folder containing pickle files
folder_path = '/content/Jan2014'

# Process each pickle file in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.pkl'):
        file_path = os.path.join(folder_path, file_name)
        process_pickle_file(file_path)


In [ ]:
import os
import h5py as h5
import numpy as np
import grid_func  # Assuming grid_func.py is available in your program folder
import matplotlib.pyplot as plt
import pickle
from sklearn.mixture import GaussianMixture

# Function to process each pickle file
def process_pickle_file(pickle_file_path):
    # Open the pickle file and read its content
    with open(pickle_file_path, 'rb') as pickle_file:
        data = pickle_file.read()

    # Deserialize the content of the pickle file
    spdct = pickle.loads(data)

    # Compute flag and level using grid_func.dcc_flag()
    flag, level = grid_func.dcc_flag(spdct['bt1sp'], spdct['bt2sp'], spdct['bt3sp'], spdct['bt4sp'], spdct['bt5sp'], spdct['stznsp'])
    spdct['dcc'] = flag
    spdct['level'] = level

    # Filter data where stznsp and bt2sp are not equal to -9999.0
    noval = -9999.0
    ##land_idx = np.where((spdct['stznsp'] != noval) * (spdct['bt2sp'] != noval) * (spdct['mask'] == 1))
    ocean_idx = np.where((spdct['stznsp'] != noval) * (spdct['bt2sp'] != noval) * (spdct['mask'] == 0)*(spdct['bt1sp'] <= 250))
    benchmark_data = np.where(spdct['dcc'] != noval)

    # Create train and test datasets for Land
    #train_Land = {}
    #train_Land['stznsp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    #train_Land['bt1sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    #train_Land['bt2sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    #train_Land['bt3sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    #train_Land['bt4sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    #train_Land['bt5sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    #train_Land['mask'] = np.ones(np.shape(spdct['stznsp'])) * noval


    #train_Land['stznsp'] = spdct['stznsp'][land_idx]
    #train_Land['bt1sp'] = spdct['bt1sp'][land_idx]
    #train_Land['bt2sp'] = spdct['bt2sp'][land_idx]
    #train_Land['bt3sp'] = spdct['bt3sp'][land_idx]
    #train_Land['bt4sp'] = spdct['bt4sp'][land_idx]
    #train_Land['bt5sp'] = spdct['bt5sp'][land_idx]
    #train_Land['mask'] = spdct['mask'][land_idx]

    #test_Land = {}
    #test_Land['dcc'] = spdct['dcc'][land_idx]
    #test_Land['level'] = spdct['level'][land_idx]

    # Prepare features for Gaussian Mixture Model
    #features = ['stznsp', 'bt1sp', 'bt2sp', 'bt3sp', 'bt4sp', 'bt5sp']
    #X_train_land = np.column_stack([train_Land[feature] for feature in features])

    # Initialize and train Gaussian Mixture Model
    #n_components = 3 # Number of components (clusters)
    #gmm = GaussianMixture(n_components=n_components, covariance_type='full')
    #gmm.fit(X_train_land)

    # Predict cluster assignments for the training data
    #train_predict_land = gmm.predict(X_train_land)

    # Prepare predicted data
    #predicted_land = np.ones(np.shape(spdct['stznsp'])) * noval
    #flat_idx_land = [(land_idx[0][i], land_idx[1][i]) for i in range(len(land_idx[0]))]
    #for (i, j), label in zip(flat_idx_land, train_predict_land):
        #predicted_land[i, j] = label

    # Mask the predicted data to handle -9999 values
    #masked_data_land = np.ma.masked_equal(predicted_land, noval)


    # Create train and test datasets for Ocean
    train_ocean = {}
    train_ocean['stznsp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train_ocean['bt1sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train_ocean['bt2sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train_ocean['bt3sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train_ocean['bt4sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train_ocean['bt5sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train_ocean['mask'] = np.ones(np.shape(spdct['stznsp'])) * noval


    train_ocean['stznsp'] = spdct['stznsp'][ocean_idx]
    train_ocean['bt1sp'] = spdct['bt1sp'][ocean_idx]
    train_ocean['bt2sp'] = spdct['bt2sp'][ocean_idx]
    train_ocean['bt3sp'] = spdct['bt3sp'][ocean_idx]
    train_ocean['bt4sp'] = spdct['bt4sp'][ocean_idx]
    train_ocean['bt5sp'] = spdct['bt5sp'][ocean_idx]
    train_ocean['mask'] = spdct['mask'][ocean_idx]

    test_ocean = {}
    test_ocean['dcc'] = spdct['dcc'][benchmark_data]
    test_ocean['level'] = spdct['level'][benchmark_data]

    # Prepare features for Gaussian Mixture Model
    features = ['stznsp', 'bt1sp', 'bt2sp', 'bt3sp', 'bt4sp', 'bt5sp']
    X_train_ocean = np.column_stack([train_ocean[feature] for feature in features])

    # Initialize and train Gaussian Mixture Model
    n_components = 3 # Number of components (clusters)
    gmm = GaussianMixture(n_components=n_components, covariance_type='full')
    gmm.fit(X_train_ocean)

    # Predict cluster assignments for the training data
    train_predict_ocean = gmm.predict(X_train_ocean)
    # Prepare predicted data
    predicted_ocean = np.ones(np.shape(spdct['stznsp'])) * noval
    flat_idx_ocean = [(ocean_idx[0][i], ocean_idx[1][i]) for i in range(len(ocean_idx[0]))]
    for (i, j), label in zip(flat_idx_ocean, train_predict_ocean):
        predicted_ocean[i, j] = label

    # Prepare test data
    test_ocean = np.ones(np.shape(spdct['stznsp'])) * noval
    test_idx_ocean = [(benchmark_data[0][i], benchmark_data[1][i]) for i in range(len(benchmark_data[0]))]
    for (i, j), label in zip(test_idx_ocean, test_ocean['dcc']):
        predicted_ocean[i, j] = label

    # Mask the predicted data to handle -9999 values
    masked_data_ocean = np.ma.masked_equal(predicted_ocean, noval)

    # Create a figure with two subplots side by side
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    # Plot data 1 (land) on the first subplot (ax1)
    im1 = ax1.imshow(masked_data_ocean, cmap='viridis', interpolation='nearest')
    ax1.set_title('Ocean Data')
    # Create colorbar for data 1 (land)
    cbar1 = fig.colorbar(im1, ax=ax1, ticks=[0, 1, 2])
    cbar1.set_ticklabels(['0', '1', '2'])

    # Plot data 2 (ocean) on the second subplot (ax2)
    im2 = ax2.imshow(spdct['dcc'], cmap='viridis', interpolation='nearest')
    ax2.set_title('DCC Benchmark Data')
    # Create colorbar for data 2 (ocean)
    cbar2 = fig.colorbar(im2, ax=ax2, ticks=[0, 1])
    cbar2.set_ticklabels(['0', '1'])

    # Customize titles based on file name
    file_name_without_extension = os.path.splitext(os.path.basename(pickle_file_path))[0]
    ax1.set_title(f'Ocean Data')
    ax2.set_title(f'DCC Benchmark Data')

    # Save the entire figure with both subplots
    save_path = os.path.join('/content/Plots_Jan2014', f"{os.path.splitext(file_name)[0]}.png")
    plt.savefig(save_path)
    plt.close()  # Close plot to free up memory

# Specify the path to the folder containing pickle files
folder_path = '/content/Jan2014'

# Process each pickle file in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.pkl'):
        file_path = os.path.join(folder_path, file_name)
        process_pickle_file(file_path)


IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

In [ ]:
import os
import h5py as h5
import numpy as np
import grid_func  # Assuming grid_func.py is available in your program folder
import matplotlib.pyplot as plt
import pickle
from sklearn.mixture import GaussianMixture

# Function to process each pickle file
def process_pickle_file(pickle_file_path):
    # Open the pickle file and read its content
    with open(pickle_file_path, 'rb') as pickle_file:
        data = pickle_file.read()

    # Deserialize the content of the pickle file
    spdct = pickle.loads(data)

    # Compute flag and level using grid_func.dcc_flag()
    flag, level = grid_func.dcc_flag(spdct['bt1sp'], spdct['bt2sp'], spdct['bt3sp'], spdct['bt4sp'], spdct['bt5sp'], spdct['stznsp'])
    spdct['dcc'] = flag
    spdct['level'] = level

    # Filter data where stznsp and bt2sp are not equal to -9999.0
    noval = -9999.0
    ocean_idx = np.where((spdct['stznsp'] != noval) * (spdct['bt2sp'] != noval) * (spdct['mask'] == 0))



    # Create train and test datasets for Ocean
    train_ocean = {}
    train_ocean['stznsp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train_ocean['bt1sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train_ocean['bt2sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train_ocean['bt3sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train_ocean['bt4sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train_ocean['bt5sp'] = np.ones(np.shape(spdct['stznsp'])) * noval
    train_ocean['mask'] = np.ones(np.shape(spdct['stznsp'])) * noval


    train_ocean['stznsp'] = spdct['stznsp'][ocean_idx]
    train_ocean['bt1sp'] = spdct['bt1sp'][ocean_idx]
    train_ocean['bt2sp'] = spdct['bt2sp'][ocean_idx]
    train_ocean['bt3sp'] = spdct['bt3sp'][ocean_idx]
    train_ocean['bt4sp'] = spdct['bt4sp'][ocean_idx]
    train_ocean['bt5sp'] = spdct['bt5sp'][ocean_idx]
    train_ocean['mask'] = spdct['mask'][ocean_idx]

    test_ocean = {}
    test_ocean['dcc'] = spdct['dcc'][land_idx]
    test_ocean['level'] = spdct['level'][land_idx]

    # Prepare features for Gaussian Mixture Model
    features = ['stznsp', 'bt1sp', 'bt2sp', 'bt3sp', 'bt4sp', 'bt5sp']
    X_train_ocean = np.column_stack([train_ocean[feature] for feature in features])

    # Initialize and train Gaussian Mixture Model
    n_components = 3 # Number of components (clusters)
    gmm = GaussianMixture(n_components=n_components, covariance_type='full')
    gmm.fit(X_train_ocean)

    # Predict cluster assignments for the training data
    train_predict_ocean = gmm.predict(X_train_ocean)
    # Prepare predicted data
    predicted_ocean = np.ones(np.shape(spdct['stznsp'])) * noval
    flat_idx_ocean = [(ocean_idx[0][i], ocean_idx[1][i]) for i in range(len(ocean_idx[0]))]
    for (i, j), label in zip(flat_idx_ocean, train_predict_ocean):
        predicted_ocean[i, j] = label

    # Mask the predicted data to handle -9999 values
    masked_data_ocean = np.ma.masked_equal(predicted_ocean, noval)

    # Plotting
    plt.imshow(masked_data_ocean, cmap='viridis', interpolation='nearest')

    # Create colorbar
    cbar = plt.colorbar()
    cbar.set_ticks([0, 1, 2])  # Set ticks for valid data (0 and 1)
    cbar.set_ticklabels(['0', '1', '2'])  # Set labels for ticks

    # Extract file name from path
    file_name = os.path.basename(pickle_file_path)
    plt.title(f"Cluster Plot - {file_name}")

    # Save plot to a file (adjust the saving path as needed)
    save_path = os.path.join('/content/Plots_Jan2014', f"{os.path.splitext(file_name)[0]}.png")
    plt.savefig(save_path)
    plt.close()  # Close plot to free up memory


    # Specify the path to the folder containing pickle files
    folder_path = '/content/Jan2014'

    # Process each pickle file in the folder
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.pkl'):
            file_path = os.path.join(folder_path, file_name)
            process_pickle_file(file_path)
